##Partie II : Scraping des avis google maps


L'objectif de cette partie était de récupérer automatiquement les avis Google Maps des pharmacies françaises à partir d'un fichier CSV, puis sauvegarder les résultats pour éffectutuer une étude sur l'e-réputation des pharmacies de France métropolitaine

Pour optimiser notre travail nous avons décidé de nous répartir les différentes régions

In [1]:
import pandas as pd
import geopandas as gpd
#from shapely import wkt
#import matplotlib.pyplot as plt

In [2]:
pharm_final = pd.read_csv("/content/pharma_france-2.csv")

In [3]:
pharm_final["nom"].unique()

array(['Île-de-France', 'Normandie', 'Hauts-de-France', 'Grand Est',
       'Auvergne-Rhône-Alpes', 'Pays de la Loire', 'Occitanie',
       'Centre-Val de Loire', "Provence-Alpes-Côte d'Azur",
       'Nouvelle-Aquitaine', 'Bourgogne-Franche-Comté', 'Bretagne', nan,
       'Corse'], dtype=object)

In [4]:
regions_cibles = [
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Pays de la Loire",
    "Provence-Alpes-Côte d'Azur"
]

pharma_cibles = pharm_final[
    pharm_final["nom"].isin(regions_cibles)
]

print(len(pharma_cibles))
pharma_cibles["nom"].value_counts()

6616


,count
nom,
Nouvelle-Aquitaine,1994
Occitanie,1903
Provence-Alpes-Côte d'Azur,1690
Pays de la Loire,1029


In [5]:
pharm_final["name"].isna().sum()

np.int64(1265)

In [6]:
pharm_final["name"].isna().mean() * 100

np.float64(6.610231488739092)

In [7]:
pharma_scraping = pharma_cibles[
    pharma_cibles["name"].notna()
].copy()

In [8]:
print(len(pharma_scraping))
pharma_scraping["nom"].value_counts()

6112


,count
nom,
Nouvelle-Aquitaine,1837
Occitanie,1784
Provence-Alpes-Côte d'Azur,1586
Pays de la Loire,905


In [9]:
df_na = pharma_scraping[
    pharma_scraping["nom"] == "Nouvelle-Aquitaine"
].copy()

df_occ = pharma_scraping[
    pharma_scraping["nom"] == "Occitanie"
].copy()

df_pdl = pharma_scraping[
    pharma_scraping["nom"] == "Pays de la Loire"
].copy()

df_paca = pharma_scraping[
    pharma_scraping["nom"] == "Provence-Alpes-Côte d'Azur"
].copy()

In [10]:
print(len(df_na))
print(len(df_occ))
print(len(df_paca))
print(len(df_pdl))

1837
1784
1586
905


Nous allons créer des fichiers csv pour chaque région afin d'aptimiser le travail

In [11]:
df_na.to_csv("nouvelle_aquitaine.csv", index=False)

df_occ.to_csv("occitanie.csv", index=False)

df_pdl.to_csv("pays_loire.csv", index=False)

df_paca.to_csv("paca.csv", index=False)

Requête google maps

Nous allons faire une étude pilote sur un échantillon de 20 pharmacies sur notre df pays de la loire

In [12]:
df_pdl.columns

Index(['FID', 'name', 'adresse_complete', 'ville', 'code_postal', 'nom',
       'longitude', 'latitude', 'osm_id'],
      dtype='object')

In [13]:
df_pdl.dtypes

,0
FID,object
name,object
adresse_complete,object
ville,object
code_postal,float64
nom,object
longitude,float64
latitude,float64
osm_id,int64


#Srapping par région

Les bibliothèque utilisées étaient :
1. pandas pour lire et écrire les fichiers CSV
2. playwright pour contrôler un navigateur pour accéder aux pages Google Maps (contenu chargé en JavaScript)
3. nest_asyncio,pour permettre l'exécution asynchrone dans Google Colab
4. re pour Extraire des informations avec des expressions régulières (dates, notes, etc.)
5. unicodedata pour normaliser le texte (supprimer les accents pour comparer les régions)

L'URL utilisé dans le script était : [google.fr](https://www.google.fr/maps/search/nom+pharmacie+ville+pharmacie)


# Les principales étapes du script :
1. Utilisation d'une fonction keep alive  pour éviviter que colab se déconnecte pendant le scraping (toutes les 60 secondes (60000 ms), il clique automatiquement sur le bouton de connexion Colab)
2. Charger le CSV des pharmacies et vérifier les colonnes nécessaires
3. Filtrer par région demandée (ex : Provence-Alpes-Côte d'Azur)
4. Pour chaque pharmacie :
* Ouvrir Google Maps dans un navigateur automatisé
* Accepter les cookies
* Récupérer la note moyenne
* Cliquer sur l'onglet "Avis"
* Faire défiler pour charger tous les avis
* Extraire : auteur, date, note, commentaire, réponse du propriétaire
* Nettoyer le texte (retirer boutons, métadonnées parasites)
* Sauvegarder dans un fichier CSV directement enregistré sur google drive
* Gérer les erreurs : si il y avait trop de plantages, redémarrer le navigateur
* Permettre la reprise : un fichier checkpoint enregistre la dernière pharmacie traitée pour qu'au remarrage d'une session le travail ne reprenne pas à zéros

#Chaque ligne de notre fichier csv de sortie contenait:

* FID : Identifiant unique de la pharmacie
pharmacie,
* Nom de la pharmacie
* ville : Ville
* region : Région
* note_moyenne_pharmacie : Note globale Google Maps
* date_avis : Date relative de l'avis
* note_avis : Note individuelle (1-5)
* auteur : Nom de l'auteur
* texte_avis : Commentaire nettoyé
* longueur_texte : Nombre de caractères
* nb_mots : Nombre de mots
* reponse_proprietaire : Texte de la réponse du propriétaire
* reponse_proprietaire_detectee : 1 si une réponse existe, 0 sinon

In [14]:
!pip install playwright
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 19.3 MB/s eta 0:00:00
177 MiB [] 0% 0.0s177 MiB [] 0% 23.4s177 MiB [] 0% 13.6s177 MiB [] 0% 13.0s177 MiB [] 1% 5.7s177 MiB [] 1% 5.6s177 MiB [] 2% 5.0s177 MiB [] 2% 4.6s177 MiB [] 3% 4.4s177 MiB [] 3% 4.1s177 MiB [] 4% 4.0s177 MiB [] 5% 3.9s177 MiB [] 6% 3.9s177 MiB [] 6% 4.0s177 MiB [] 6% 4.2s177 MiB [] 6% 4.3s177 MiB [] 7% 3.9s177 MiB [] 8% 3.7s177 MiB [] 9% 3.4s177 MiB [] 10% 3.2s177 MiB [] 11% 3.0s177 MiB [] 12% 2.9s177 MiB [] 13% 2.8s177 MiB [] 14% 2.7s177 MiB [] 14% 2.6s177 MiB [] 15% 2.5s177 MiB [] 16% 2.4s177 MiB [] 17% 2.4s177 MiB [] 18% 2.3s177 MiB [] 19% 2.2s177 MiB [] 20% 2.2s177 MiB [] 21% 2.1s177 MiB [] 22% 2.1s177 MiB [] 22% 2.2s177 MiB [] 23% 2.2s177 MiB [] 24% 2.1s177 MiB [] 25% 2.0s177 MiB [] 26% 2.0s177 MiB [] 28% 1.9s177 MiB [] 29% 1.8s177 MiB [] 30% 1.7s177 MiB [] 31% 1.7s177 MiB [] 32% 1.6s177 MiB [] 33% 1.6s177 MiB [] 34% 1.6s177 MiB [] 35% 1.5s177 MiB [] 36% 1.5s177 MiB [] 37% 1.4s177 MiB [] 38% 1.4s177 Mi

D'abord il faut faire ça

Ici nous allons faire un keep alive

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
from IPython.display import HTML

HTML("""
<script>
function ClickConnect(){
  console.log("Colab keep alive");
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}
setInterval(ClickConnect, 60000);
</script>
""")


Ici nous allons démarrer le scraping par region

In [17]:
#import os

#os.remove("/content/avis_pharmacies_Occitanie.csv")

In [18]:
import os
import re
import json
import shutil
import asyncio
import random
import pandas as pd
import unicodedata
def normaliser(texte):
    if pd.isna(texte):
        return ""

    texte = str(texte)

    # Uniformiser les apostrophes
    texte = texte.replace("’", "'")

    # Supprimer les accents
    texte = unicodedata.normalize("NFKD", texte)
    texte = texte.encode("ASCII", "ignore").decode("utf-8")

    # Nettoyage
    return texte.strip().lower()

import nest_asyncio
from datetime import datetime
from urllib.parse import quote_plus
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

# Dependances systeme Playwright
!apt-get update -y
!apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libgdk-pixbuf2.0-0 libgtk-3-0 libgbm-dev

# Si besoin :
# !pip install playwright
# !playwright install chromium

nest_asyncio.apply()


# 1. PARAMETRES GENERAUX


CSV_SOURCE = "/content/pharma_france-2.csv"

regions_cibles = [
    #"Occitanie",
    # "Nouvelle-Aquitaine",
    # "Pays de la Loire",
     "Provence-Alpes-Côte d'Azur"
]

# None = toute la region
NB_MAX_PHARMACIES_PAR_REGION = None # à remplacer par None si ça marche

# None = autant d'avis que possible
NB_MAX_AVIS_PAR_PHARMACIE = None # à remplacer par None si ça marche

MAX_SCROLLS = 80 # à remplacer par 80 si ça marche
SCROLL_PAUSE_MS = 2200
DELAI_ENTRE_PHARMACIES_MIN = 2.5
DELAI_ENTRE_PHARMACIES_MAX = 5.5

TIMEOUT_GOTO = 60000
TIMEOUT_WAIT_STANDARD = 5000
TIMEOUT_WAIT_SHORT = 2000

# Nombre d'erreurs critiques consecutives avant redemarrage complet du navigateur
MAX_ERREURS_CONSECUTIVES = 5

# Sauvegarde / logs
DRIVE_BASE = "/content/drive/MyDrive/scraping_pharmacies"
DRIVE_LOGS = os.path.join(DRIVE_BASE, "logs")
DRIVE_CHECKPOINTS = os.path.join(DRIVE_BASE, "checkpoints")

PREFIX_SORTIE = "avis_pharmacies"
PREFIX_LOG = "log_scraping"
PREFIX_CHECKPOINT = "checkpoint"

# Fichiers locaux /content
#LOCAL_BASE = "/content"


# 2. CHARGEMENT DES DONNEES + RESOLUTION DES REGIONS

COLONNES_OBLIGATOIRES = [
    "FID",
    "name",
    "ville",
    "nom",
]
def normaliser_texte(valeur):
    """
    Normalise un texte pour les comparaisons robustes :
    - conversion en str
    - trim
    - minuscules
    - suppression des accents
    - homogénéisation des apostrophes
    - remplacement des tirets par espaces
    - réduction des espaces multiples
    """
    if pd.isna(valeur):
        return None
    valeur = str(valeur).strip().lower()
    valeur = valeur.replace("’", "'")
    valeur = valeur.replace("-", " ")
    valeur = unicodedata.normalize("NFKD", valeur)
    valeur = "".join(c for c in valeur if not unicodedata.combining(c))
    valeur = re.sub(r"\s+", " ", valeur).strip()
    return valeur
def verifier_colonnes_obligatoires(df, colonnes_obligatoires):
    """
    Vérifie que toutes les colonnes nécessaires sont présentes dans le CSV source.
    Arrête explicitement le script si une colonne manque.
    """
    colonnes_manquantes = [col for col in colonnes_obligatoires if col not in df.columns]
    if colonnes_manquantes:
        raise ValueError(
            "Le fichier source ne contient pas toutes les colonnes obligatoires.\n"
            f"Colonnes manquantes : {colonnes_manquantes}\n"
            f"Colonnes disponibles : {list(df.columns)}"
        )
def charger_fichier_source(csv_source):
    """
    Charge le CSV source des pharmacies et valide immédiatement sa structure.
    """
    if not os.path.exists(csv_source):
        raise FileNotFoundError(f"Fichier source introuvable : {csv_source}")
    df = pd.read_csv(csv_source)
    verifier_colonnes_obligatoires(df, COLONNES_OBLIGATOIRES)
    print(f" Fichier source chargé : {csv_source}")
    print(f" Nombre total de lignes : {len(df)}")
    return df
def construire_mapping_regions(df):
    """
    Construit un mapping robuste :
    region_normalisee -> region_reelle_du_csv
    Exemple :
    'provence alpes cote d azur' -> 'Provence-Alpes-Côte d'Azur'
    """
    regions_reelles = (
        df["nom"]
        .dropna()
        .astype(str)
        .map(lambda x: x.strip())
        .unique()
        .tolist()
    )
    mapping = {}
    for region in regions_reelles:
        cle = normaliser_texte(region)
        if cle not in mapping:
            mapping[cle] = region
    return mapping
def afficher_regions_disponibles(mapping_regions):
    """
    Affiche proprement les régions réellement présentes dans le CSV.
    """
    print("\n Régions disponibles dans le fichier source :")
    for region in sorted(mapping_regions.values()):
        print(f"   - {region}")
def resoudre_regions_demandees(regions_cibles, mapping_regions):
    """
    Résout les régions demandées vers leurs libellés exacts présents dans le CSV.
    Arrête proprement si une région demandée n'existe pas.
    """
    regions_resolues = []
    regions_introuvables = []
    for region in regions_cibles:
        cle = normaliser_texte(region)
        if cle in mapping_regions:
            regions_resolues.append(mapping_regions[cle])
        else:
            regions_introuvables.append(region)
    if regions_introuvables:
        print("\n Certaines régions demandées sont introuvables dans le CSV :")
        for region in regions_introuvables:
            print(f"   - {region}")
        afficher_regions_disponibles(mapping_regions)
        raise ValueError(
            "Au moins une région demandée n'existe pas dans le fichier source. "
            "Le scraping est arrêté pour éviter de continuer avec un DataFrame vide."
        )
    print("\n Régions demandées validées :")
    for region in regions_resolues:
        print(f"   - {region}")
    return regions_resolues
def preparer_dataframe_region(df_source, region_reelle, conserver_nom_valide=True):
    """
    Prépare le DataFrame d'une région :
    - filtre la région réelle
    - conserve éventuellement seulement les pharmacies avec 'name' valide
    - ajoute une colonne 'Region' homogène si nécessaire
    - retourne directement le DataFrame prêt pour le scraper
    """
    df_region = df_source[df_source["nom"] == region_reelle].copy()
    if conserver_nom_valide:
        df_region = df_region[df_region["name"].notna()].copy()
        df_region = df_region[df_region["name"].astype(str).str.strip() != ""].copy()
    df_region["Region"] = df_region["nom"]
    return df_region
def afficher_resume_regions(df_source, regions_resolues):
    """
    Affiche le nombre de pharmacies trouvées par région avant lancement du scraping.
    """
    print("\n Résumé des régions à traiter :")
    for region in regions_resolues:
        nb_total = (df_source["nom"] == region).sum()
        nb_valides = len(
            df_source[
                (df_source["nom"] == region)
                & (df_source["name"].notna())
                & (df_source["name"].astype(str).str.strip() != "")
            ]
        )
        print(f"   - {region} : {nb_total} lignes, {nb_valides} pharmacies avec nom exploitable")
# Chargement unique du fichier source
df_source = charger_fichier_source(CSV_SOURCE)
# Construction du mapping de régions
mapping_regions = construire_mapping_regions(df_source)
# Résolution robuste des régions demandées
regions_cibles_resolues = resoudre_regions_demandees(regions_cibles, mapping_regions)
# Affichage du résumé avant scraping
afficher_resume_regions(df_source, regions_cibles_resolues)


# 3. OUTILS GENERAUX


def maintenant_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def nettoyer_espaces(txt):
    if txt is None:
        return None
    txt = str(txt).replace("\xa0", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt if txt else None


def nettoyer_url_markdown(url):
    if url is None:
        return None
    url = str(url).strip()
    match = re.match(r"^\[.*?\]\((https?://.*?)\)$", url)
    if match:
        return match.group(1)
    return url


def chemin_csv_region(region):
    safe = region.replace(" ", "_").replace("-", "_")
    return os.path.join(DRIVE_BASE, f"{PREFIX_SORTIE}_{safe}.csv")


def chemin_log_region(region):
    safe = region.replace(" ", "_").replace("-", "_")
    return os.path.join(DRIVE_BASE, f"{PREFIX_LOG}_{safe}.csv")


def chemin_checkpoint_region(region):
    safe = region.replace(" ", "_").replace("-", "_")
    return os.path.join(DRIVE_BASE, f"{PREFIX_CHECKPOINT}_{safe}.json")


#def copier_fichier_vers_drive(path_local, sous_dossier_drive=None):
 #   if not os.path.isfile(path_local):
  #      return
#
 #   if not os.path.isdir(DRIVE_BASE):
  #      return
#
 #   destination_dir = DRIVE_BASE if sous_dossier_drive is None else sous_dossier_drive
  #  os.makedirs(destination_dir, exist_ok=True)
#
 #   destination = os.path.join(destination_dir, os.path.basename(path_local))
  #  shutil.copy2(path_local, destination)
   # print(f" Copie Drive OK : {destination}")


def ecrire_log_csv(path_log, ligne_dict):
    df_log = pd.DataFrame([ligne_dict])
    header = not os.path.exists(path_log)
    df_log.to_csv(
        path_log,
        mode="a",
        header=header,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )


def sauvegarder_checkpoint(path_checkpoint, data):
    with open(path_checkpoint, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def charger_checkpoint(path_checkpoint):
    if not os.path.exists(path_checkpoint):
        return None
    try:
        with open(path_checkpoint, "r", encoding="utf-8") as f:
            return json.load(f)
    except:
        return None


def nettoyer_note_moyenne(note_str):
    if not note_str:
        return None

    match = re.search(r"(\d+[.,]?\d*)", str(note_str))
    if match:
        return match.group(1).replace(",", ".")
    return None


def extraire_date_avis(texte):
    if not texte:
        return None

    patterns = [
        r"(modifié\s+il y a\s+\d+\s+(?:jour|jours|semaine|semaines|mois|an|ans))",
        r"(il y a\s+\d+\s+(?:jour|jours|semaine|semaines|mois|an|ans))",
        r"(il y a\s+une\s+(?:semaine|année))",
        r"(il y a\s+un\s+(?:jour|mois|an))",
        r"(modified\s+(?:\d+\s+)?(?:day|days|week|weeks|month|months|year|years)\s+ago)",
        r"((?:\d+\s+)?(?:day|days|week|weeks|month|months|year|years)\s+ago)",
        r"(a\s+(?:day|week|month|year)\s+ago)"
    ]

    texte = nettoyer_espaces(texte)

    for pattern in patterns:
        match = re.search(pattern, texte, flags=re.IGNORECASE)
        if match:
            date_avis = nettoyer_espaces(match.group(1))
            date_avis = date_avis.replace(" an", " ans") if date_avis == "il y a 1 an" else date_avis
            return date_avis

    return None


def extraire_note_avis_depuis_texte(texte):
    if not texte:
        return None

    lignes = [l.strip() for l in str(texte).split("\n") if l.strip()]
    for ligne in lignes:
        if re.fullmatch(r"[★☆]+", ligne):
            nb_etoiles = ligne.count("★")
            if 1 <= nb_etoiles <= 5:
                return nb_etoiles
    return None


def extraire_auteur(texte):
    if not texte:
        return None

    lignes = [l.strip() for l in str(texte).split("\n") if l.strip()]
    if not lignes:
        return None

    premiere = nettoyer_espaces(lignes[0])
    if premiere and len(premiere) <= 120:
        return premiere
    return None


def nettoyer_commentaire(texte_complet, date_avis=None, auteur=None):
    """
    Nettoyage orienté Google Maps :
    - retire icones parasites
    - retire etoiles
    - retire date relative
    - retire 'Visité en ...'
    - retire 'J'aime', 'Partager'
    - coupe avant 'Réponse du propriétaire'
    """
    if not texte_complet:
        return None

    txt = str(texte_complet)

    # Enlever quelques caracteres parasites frequents
    txt = re.sub(r"[]+", " ", txt)

    # Normaliser espaces / retours
    txt = txt.replace("\xa0", " ")
    txt = re.sub(r"\r", "\n", txt)
    txt = re.sub(r"\n+", "\n", txt)

    lignes = [l.strip() for l in txt.split("\n") if l.strip()]
    if not lignes:
        return None

    txt_flat = " ".join(lignes)
    txt_flat = nettoyer_espaces(txt_flat)

    # Couper avant reponse proprietaire
    txt_flat = re.split(r"réponse du propriétaire|owner response", txt_flat, flags=re.IGNORECASE)[0]

    # Retirer auteur si present au debut
    if auteur:
        txt_flat = re.sub(rf"^{re.escape(auteur)}\s*", "", txt_flat, flags=re.IGNORECASE)

    # Retirer suites d'etoiles unicode
    txt_flat = re.sub(r"[★☆]+", " ", txt_flat)

    # Retirer metadata frequentes
    txt_flat = re.sub(r"\bLocal Guide\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\b\d+\s+avis\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\b\d+\s+reviews?\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\b\d+\s+photos?\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\b\d+\s+contributions?\b", " ", txt_flat, flags=re.IGNORECASE)

    # Retirer la date relative si on l'a extraite
    if date_avis:
        txt_flat = re.sub(re.escape(date_avis), " ", txt_flat, flags=re.IGNORECASE)

    # Retirer autres formes de dates relatives si elles trainent encore
    txt_flat = re.sub(r"\bmodifié\s+il y a\s+\d+\s+(jour|jours|semaine|semaines|mois|an|ans)\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bil y a\s+\d+\s+(jour|jours|semaine|semaines|mois|an|ans)\b", " ", txt_flat, flags=re.IGNORECASE)

    # Retirer "Visité en ..."
    txt_flat = re.sub(r"Visité en [^.]*?(?=J'aime|Partager|$)", " ", txt_flat, flags=re.IGNORECASE)

    # Retirer boutons / UI
    txt_flat = re.sub(r"\bJ'aime\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bPartager\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bLike\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bShare\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bPlus\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bMore\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bLire la suite\b", " ", txt_flat, flags=re.IGNORECASE)
    txt_flat = re.sub(r"\bSee more\b", " ", txt_flat, flags=re.IGNORECASE)

    # Retirer petits compteurs isoles
    txt_flat = re.sub(r"\s+\d+\s+", " ", txt_flat)
    # Retirer les résidus du type "· s", "· m", "· h"
    txt_flat = re.sub(r"^[·•]\s*[smhd]\s+", "", txt_flat)

    # Retirer les résidus du type "· 2 s"
    txt_flat = re.sub(r"^[·•]\s*\d+\s*[smhd]\s+", "", txt_flat)

    # Nettoyage ponctuation en début de commentaire
    txt_flat = re.sub(r"^[·•\-\s]+", "", txt_flat)

    txt_flat = nettoyer_espaces(txt_flat)

    #Pour plus de robustesse
    txt_flat = nettoyer_espaces(txt_flat)

    # Si le commentaire commence encore par un résidu
    txt_flat = re.sub(
    r"^(?:[·•]|Local Guide|avis|reviews?|photos?|\d+)\s+",
    "",
    txt_flat,
    flags=re.IGNORECASE
    )

    txt_flat = nettoyer_espaces(txt_flat)

    # Supprime un résidu isolé en début de commentaire
    txt_flat = re.sub(
    r"^[smhd]\s+",
    "",
    txt_flat,
    flags=re.IGNORECASE
    )

    txt_flat = nettoyer_espaces(txt_flat)

    return txt_flat if txt_flat else None
    #Rajoutons les fct de récupération des commentaires du propriétaire

def extraire_reponse_proprietaire(texte_complet):
    """
    Extrait le texte de la réponse du propriétaire si elle existe.
    Retourne None si aucune réponse n'est présente.
    """

    if not texte_complet:
        return None

    texte = str(texte_complet)

    # Recherche de "Réponse du propriétaire" ou "Owner response"
    match = re.search(
        r"(?:Réponse du propriétaire|Owner response)(.*)",
        texte,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return None

    contenu = match.group(1).strip()

    # Découpage en lignes
    lignes = [l.strip() for l in contenu.split("\n") if l.strip()]

    if not lignes:
        return None

    # Si la première ligne est une date ("il y a 2 semaines"),
    # on l'enlève car elle sera récupérée dans une autre fonction.
    if extraire_date_avis(lignes[0]):
        lignes = lignes[1:]

    if not lignes:
        return None

    return nettoyer_espaces(" ".join(lignes))


# Extraction des dates des réponses du propriétaire
def extraire_date_reponse_proprietaire(texte_complet):
    """
    Extrait la date de la réponse du propriétaire.
    Exemple :
        Réponse du propriétaire
        il y a 2 semaines

    Retourne None si aucune date n'est trouvée.
    """

    if not texte_complet:
        return None

    texte = str(texte_complet)

    match = re.search(
        r"(?:Réponse du propriétaire|Owner response)(.*)",
        texte,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return None

    contenu = match.group(1)

    lignes = [l.strip() for l in contenu.split("\n") if l.strip()]

    if not lignes:
        return None

    # On regarde uniquement les premières lignes
    for ligne in lignes[:3]:
        date = extraire_date_avis(ligne)
        if date:
            return date

    return None
#Fin des deux nouvelles fonction ajoutées


def dedoublonner_avis(liste_avis):
    uniques = []
    vus = set()

    for avis in liste_avis:
        cle = (
            str(avis.get("FID")),
            nettoyer_espaces(avis.get("date_avis")),
            avis.get("note_avis"),
            nettoyer_espaces(avis.get("texte_avis"))
        )
        if cle not in vus:
            vus.add(cle)
            uniques.append(avis)

    return uniques

async def extraire_note_avis_depuis_bloc(avis_bloc):
    try:
        elements = await avis_bloc.locator("[aria-label*='étoile'], [aria-label*='star']").all()
        for el in elements:
            aria = await el.get_attribute("aria-label")
            if aria:
                match = re.search(r"(\d+)", aria)
                if match:
                    note = int(match.group(1))
                    if 1 <= note <= 5:
                        return note
    except:
        pass
    return None


async def cliquer_sur_plus_si_present(page):
    selecteurs = [
        "button:has-text('Plus')",
        "button:has-text('More')",
        "button:has-text('Lire la suite')",
        "button[aria-label*='Plus']",
        "button[aria-label*='More']"
    ]

    total_clicks = 0

    for sel in selecteurs:
        try:
            boutons = page.locator(sel)
            count = await boutons.count()
            for i in range(min(count, 100)):
                try:
                    await boutons.nth(i).click(timeout=1200)
                    total_clicks += 1
                    await page.wait_for_timeout(120)
                except:
                    pass
        except:
            pass

    if total_clicks > 0:
        print(f"🔓 Avis deplies : {total_clicks}")


async def faire_defiler_les_avis(page, max_scrolls=80, pause_ms=2200, max_avis=None):
    selecteurs_zone = [
        "div[role='feed']",
        "div.m6QErb[role='region']",
        "div.m6QErb.DxyBCb.kA9KIf.dS8AEf",
        "div[aria-label*='Avis']",
        "div[aria-label*='Reviews']"
    ]

    zone_scroll = None

    for sel in selecteurs_zone:
        loc = page.locator(sel)
        try:
            if await loc.count() > 0:
                zone_scroll = loc.first
                break
        except:
            pass

    if zone_scroll is None:
        print(" Zone de scroll des avis introuvable.")
        return 0

    dernier_nombre = 0
    nb_stables = 0
    dernier_total = 0

    for i in range(max_scrolls):
        try:
            avis_blocs = await page.locator("div[data-review-id]").count()
            dernier_total = avis_blocs
            print(f" Scroll {i+1}/{max_scrolls} - avis charges : {avis_blocs}")

            if max_avis is not None and avis_blocs >= max_avis:
                print(f" Limite atteinte : {max_avis} avis")
                break

            if avis_blocs == dernier_nombre:
                nb_stables += 1
            else:
                nb_stables = 0

            if nb_stables >= 3:
                print(" Plus de nouveaux avis charges.")
                break

            dernier_nombre = avis_blocs

            await zone_scroll.evaluate("(el) => { el.scrollTop = el.scrollHeight; }")
            await page.wait_for_timeout(pause_ms)

        except Exception as e:
            print(f" Erreur scroll : {e}")
            break

    return dernier_total


async def ouvrir_google_maps(page, nom_pharma, ville):
    query = quote_plus(f"{nom_pharma} {ville} pharmacie")
    url_brute = f"[google.fr](https://www.google.fr/maps/search/{query})"
    url = nettoyer_url_markdown(url_brute)

    print("URL brute =", repr(url_brute))
    print("URL finale =", repr(url))

    if not re.match(r"^https?://", url):
        raise ValueError(f"URL invalide generee : {url}")

    await page.goto(url, timeout=TIMEOUT_GOTO)
    await page.wait_for_timeout(TIMEOUT_WAIT_STANDARD)

    # Cookies
    try:
        bouton_accepter = page.locator(
            "button:has-text('Tout accepter'), button:has-text('Accept all')"
        )
        if await bouton_accepter.count() > 0:
            await bouton_accepter.first.click()
            await page.wait_for_timeout(2500)
    except:
        pass


async def extraire_note_moyenne(page):
    note_moyenne = None
    try:
        spans = await page.locator("span[role='img']").all()
        for span in spans:
            aria = await span.get_attribute("aria-label")
            if aria and ("étoile" in aria.lower() or "star" in aria.lower()):
                note_moyenne = nettoyer_note_moyenne(aria)
                break
    except:
        pass
    return note_moyenne


async def ouvrir_onglet_avis(page):
    onglet_avis = page.locator(
        "button[role='tab']:has-text('Avis'), button:has-text('Reviews')"
    )

    if await onglet_avis.count() > 0:
        await onglet_avis.first.click()
        await page.wait_for_timeout(TIMEOUT_WAIT_STANDARD)
        return True

    return False


def construire_ligne_sans_avis(fid, nom_pharma, ville, region, note_moyenne):
    return {
        "FID": fid,
        "pharmacie": nom_pharma,
        "ville": ville,
        "region": region,
        "note_moyenne_pharmacie": note_moyenne,
        "date_avis": None,
        "note_avis": None,
        "auteur": None,
        "texte_avis": "Aucun avis texte exploitable",
        "longueur_texte": 0,
        "nb_mots": 0,
        "reponse_proprietaire": None,
        "date_reponse_proprietaire": None,
        "reponse_proprietaire_detectee": 0
    }


# 4. SCRAPING LONG RUN PAR REGION

async def scraper_region_long_run(df_region, nom_region):
    fichier_csv = chemin_csv_region(nom_region)
    fichier_log = chemin_log_region(nom_region)
    fichier_checkpoint = chemin_checkpoint_region(nom_region)


    # REPRISE PAR CSV + CHECKPOINT

    print("=" * 80)
    print("Chemin CSV local :", fichier_csv)
    print("Existe localement ?", os.path.exists(fichier_csv))

    fichier_drive = os.path.join(DRIVE_BASE, os.path.basename(fichier_csv))
    print("Chemin Drive :", fichier_drive)
    print("Existe sur Drive ?", os.path.exists(fichier_drive))
    print("=" * 80)


    # RESTAURATION AUTOMATIQUE DU CSV DEPUIS DRIVE

    if not os.path.exists(fichier_csv):

      fichier_drive = os.path.join(
        DRIVE_BASE,
        os.path.basename(fichier_csv)
    )

    if os.path.exists(fichier_drive):
        #shutil.copy2(fichier_drive, fichier_csv)
        print(" CSV restauré depuis Google Drive")
    #################
    if os.path.exists(fichier_csv):
        try:
            df_sauvegarde = pd.read_csv(fichier_csv, sep=";")
            if "FID" in df_sauvegarde.columns:
                fids_deja_faits = (
                    df_sauvegarde["FID"]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            else:
                fids_deja_faits = []
        except:
            fids_deja_faits = []
    else:
        fids_deja_faits = []

    checkpoint = charger_checkpoint(fichier_checkpoint)
    checkpoint_fid = checkpoint.get("last_completed_fid") if checkpoint else None

    df_region_temp = df_region.copy()
    df_region_temp["FID"] = df_region_temp["FID"].astype(str)

    df_a_traiter = df_region_temp[
        ~df_region_temp["FID"].isin(fids_deja_faits)
    ].copy()

    total_region = len(df_region_temp)
    total_restant = len(df_a_traiter)

    print(f" Region : {nom_region}")
    print(f" Pharmacies totales region : {total_region}")
    print(f" Pharmacies deja faites selon CSV : {len(fids_deja_faits)}")
    print(f" Pharmacies restantes : {total_restant}")

    ecrire_log_csv(fichier_log, {
        "timestamp": maintenant_str(),
        "region": nom_region,
        "event_type": "START_REGION",
        "fid": checkpoint_fid,
        "pharmacie": None,
        "ville": None,
        "status": "INFO",
        "message": f"Debut / reprise region. Restant={total_restant}",
        "avis_count": None
    })

    #copier_fichier_vers_drive(fichier_log, DRIVE_LOGS)

    #if df_a_traiter.empty:
      #  print(f" Rien a faire pour {nom_region}")
       # return

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-infobars"
            ]
        )

        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            locale="fr-FR",
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )

        erreurs_consecutives = 0
        compteur_fait = 0

        for idx, row in df_a_traiter.iterrows():
            nom_pharma = nettoyer_espaces(str(row["name"]))
            ville = nettoyer_espaces(str(row["ville"]))
            fid = str(row["FID"])
            region = nettoyer_espaces(str(row["Region"]))

            page = None
            avis_collectes = []
            note_moyenne = None
            statut = "OK"
            message_log = ""
            nb_avis_sauves = 0

            try:
                page = await context.new_page()

                print("\n" + "=" * 90)
                print(f" Extraction : {nom_pharma} - {ville} - {region}")
                print(f" FID : {fid}")
                print(f" Progression restante : {total_restant - compteur_fait}")

                ecrire_log_csv(fichier_log, {
                    "timestamp": maintenant_str(),
                    "region": nom_region,
                    "event_type": "START_PHARMACY",
                    "fid": fid,
                    "pharmacie": nom_pharma,
                    "ville": ville,
                    "status": "RUNNING",
                    "message": "Debut pharmacie",
                    "avis_count": None
                })

                await ouvrir_google_maps(page, nom_pharma, ville)

                note_moyenne = await extraire_note_moyenne(page)
                print(" Note moyenne :", note_moyenne)

                onglet_ok = await ouvrir_onglet_avis(page)

                if not onglet_ok:
                    print(" Onglet Avis introuvable.")
                    message_log = "Onglet Avis introuvable"
                else:
                    total_blocs = await faire_defiler_les_avis(
                        page,
                        max_scrolls=MAX_SCROLLS,
                        pause_ms=SCROLL_PAUSE_MS,
                        max_avis=NB_MAX_AVIS_PAR_PHARMACIE
                    )

                    print(f" Blocs avis charges apres scroll : {total_blocs}")

                    await cliquer_sur_plus_si_present(page)

                    avis_blocs = await page.locator("div[data-review-id]").all()
                    print(f" Nombre final de blocs avis : {len(avis_blocs)}")

                    for i, avis in enumerate(avis_blocs):
                        try:

                            ##
                            texte_complet = await avis.inner_text()
                            ####Test
                            #html = await avis.inner_html()
                            #print("=" * 100)
                            #print(html)
                            #print("=" * 100)
                            #break
                            ###fin test
                            texte_complet = texte_complet.strip()

                            if not texte_complet:
                                continue
                            # TEMPORAIRE POUR LE TEST
                            #if "Réponse du propriétaire" in texte_complet or "Owner response" in texte_complet:
                             # print("=" * 80)
                              #print(texte_complet)
                              #print("=" * 80)
                              #break
                            #######

                            reponse_proprietaire = extraire_reponse_proprietaire(texte_complet)
                            date_reponse_proprietaire = extraire_date_reponse_proprietaire(texte_complet)

                            auteur = extraire_auteur(texte_complet)
                            date_avis = extraire_date_avis(texte_complet)

                            note_avis = await extraire_note_avis_depuis_bloc(avis)
                            if note_avis is None:
                                note_avis = extraire_note_avis_depuis_texte(texte_complet)

                            texte_avis = nettoyer_commentaire(
                                texte_complet,
                                date_avis=date_avis,
                                auteur=auteur
                            )

                            if texte_avis and len(texte_avis) >= 3:
                                ligne = {
                                    "FID": fid,
                                    "pharmacie": nom_pharma,
                                    "ville": ville,
                                    "region": region,
                                    "note_moyenne_pharmacie": note_moyenne,
                                    "date_avis": date_avis,
                                    "note_avis": note_avis,
                                    "auteur": auteur,
                                    "texte_avis": texte_avis,
                                    "longueur_texte": len(texte_avis) if texte_avis else 0,
                                    "nb_mots": len(texte_avis.split()) if texte_avis else 0,
                                    "reponse_proprietaire": reponse_proprietaire,
                                    "date_reponse_proprietaire": date_reponse_proprietaire,
                                    "reponse_proprietaire_detectee": int(
                                        bool(re.search(r"réponse du propriétaire|owner response", texte_complet, flags=re.IGNORECASE))
                                    )
                                }
                                avis_collectes.append(ligne)

                                if i < 5:
                                    print(f"   - avis {i+1}: date={date_avis} | note={note_avis} | auteur={auteur}")

                        except Exception as e:
                            print(f"Erreur bloc avis #{i+1} : {e}")

                avis_collectes = dedoublonner_avis(avis_collectes)
                nb_avis_sauves = len(avis_collectes)

                if nb_avis_sauves > 0:
                    df_temp = pd.DataFrame(avis_collectes)
                    message_log = f"{nb_avis_sauves} avis sauvegardes"
                else:
                    df_temp = pd.DataFrame([construire_ligne_sans_avis(
                        fid, nom_pharma, ville, region, note_moyenne
                    )])
                    message_log = message_log or "Aucun avis texte exploitable"

                header_condition = not os.path.exists(fichier_csv)
                df_temp.to_csv(
                    fichier_csv,
                    mode="a",
                    header=header_condition,
                    index=False,
                    sep=";",
                    encoding="utf-8-sig"
                )

                # Checkpoint apres pharmacie terminee
                sauvegarder_checkpoint(fichier_checkpoint, {
                    "timestamp": maintenant_str(),
                    "region": nom_region,
                    "last_completed_fid": fid,
                    "last_completed_pharmacy": nom_pharma,
                    "last_completed_city": ville,
                    "last_saved_reviews": nb_avis_sauves,
                    "csv_file": fichier_csv
                })

                # Copie systematique Drive
                #copier_fichier_vers_drive(fichier_csv, DRIVE_BASE)
                #copier_fichier_vers_drive(fichier_log, DRIVE_LOGS)
                #copier_fichier_vers_drive(fichier_checkpoint, DRIVE_CHECKPOINTS)

                ecrire_log_csv(fichier_log, {
                    "timestamp": maintenant_str(),
                    "region": nom_region,
                    "event_type": "END_PHARMACY",
                    "fid": fid,
                    "pharmacie": nom_pharma,
                    "ville": ville,
                    "status": "OK",
                    "message": message_log,
                    "avis_count": nb_avis_sauves
                })

                print(f" Pharmacie terminee : {nom_pharma} | avis uniques = {nb_avis_sauves}")

                erreurs_consecutives = 0
                compteur_fait += 1

                await asyncio.sleep(random.uniform(DELAI_ENTRE_PHARMACIES_MIN, DELAI_ENTRE_PHARMACIES_MAX))

            except PlaywrightTimeoutError as e:
                statut = "TIMEOUT"
                erreurs_consecutives += 1
                message_log = f"Timeout Playwright : {str(e)[:250]}"
                print(f" Timeout sur {nom_pharma} : {e}")

                ecrire_log_csv(fichier_log, {
                    "timestamp": maintenant_str(),
                    "region": nom_region,
                    "event_type": "ERROR_PHARMACY",
                    "fid": fid,
                    "pharmacie": nom_pharma,
                    "ville": ville,
                    "status": statut,
                    "message": message_log,
                    "avis_count": 0
                })

            except Exception as e:
                statut = "ERROR"
                erreurs_consecutives += 1
                message_log = str(e)[:300]
                print(f" Erreur critique sur {nom_pharma} : {e}")

                ecrire_log_csv(fichier_log, {
                    "timestamp": maintenant_str(),
                    "region": nom_region,
                    "event_type": "ERROR_PHARMACY",
                    "fid": fid,
                    "pharmacie": nom_pharma,
                    "ville": ville,
                    "status": statut,
                    "message": message_log,
                    "avis_count": 0
                })

            finally:
                #copier_fichier_vers_drive(fichier_log, DRIVE_LOGS)

                if page is not None:
                    try:
                        if not page.is_closed():
                            await page.close()
                    except:
                        pass

                # Garde-fou : trop d'erreurs consecutives => restart navigateur
                if erreurs_consecutives >= MAX_ERREURS_CONSECUTIVES:
                    print(" Trop d'erreurs consecutives. Redemarrage complet du navigateur...")
                    try:
                        await browser.close()
                    except:
                        pass

                    browser = await p.chromium.launch(
                        headless=True,
                        args=[
                            "--disable-blink-features=AutomationControlled",
                            "--disable-infobars"
                        ]
                    )

                    context = await browser.new_context(
                        viewport={"width": 1920, "height": 1080},
                        locale="fr-FR",
                        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
                    )

                    erreurs_consecutives = 0

        try:
            await browser.close()
        except:
            pass

    ecrire_log_csv(fichier_log, {
        "timestamp": maintenant_str(),
        "region": nom_region,
        "event_type": "END_REGION",
        "fid": None,
        "pharmacie": None,
        "ville": None,
        "status": "DONE",
        "message": "Region terminee",
        "avis_count": None
    })


    #copier_fichier_vers_drive(fichier_log, DRIVE_LOGS)
    #copier_fichier_vers_drive(fichier_checkpoint, DRIVE_CHECKPOINTS)

    ##############################
    if os.path.exists(fichier_csv):
      pass
        #copier_fichier_vers_drive(fichier_csv, DRIVE_BASE)

      print(f"\n🏁 REGION TERMINEE : {nom_region}")


# 5. PILOTE GLOBAL

async def lancer_le_robot_long_run():
    """
    Lance le scraping région par région avec :
    - préparation propre des données
    - logs explicites
    - arrêt propre si une région validée ne contient finalement aucune pharmacie exploitable
    """
    print("\n Démarrage du scraping multi-régions...")
    for region in regions_cibles_resolues:
        print("\n" + "=" * 100)
        print(f" Préparation de la région : {region}")
        df_region = preparer_dataframe_region(
            df_source=df_source,
            region_reelle=region,
            conserver_nom_valide=True
        )
        nb_pharmacies = len(df_region)
        print(f" Nombre de pharmacies à traiter pour {region} : {nb_pharmacies}")
        if NB_MAX_PHARMACIES_PAR_REGION is not None:
            df_region = df_region.head(NB_MAX_PHARMACIES_PAR_REGION).copy()
            print(f" Limitation active : {len(df_region)} pharmacies seront traitées")
        if df_region.empty:
            raise ValueError(
                f"La région '{region}' a été validée mais ne contient aucune pharmacie exploitable "
                "après préparation des données. Le pipeline s'arrête pour éviter un run incohérent."
            )
        print(f" Lancement effectif du scraping pour {region}")
        await scraper_region_long_run(df_region, region)
    print("\n TOUTES LES REGIONS DEMANDEES SONT TERMINEES")


# 6. LANCEMENT


await lancer_le_robot_long_run()

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,095 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Get:14 http://archive.ubu